# 21 — Cost, Latency, and Token Engineering

## Scenario
We have a large document (e.g., an entire codebase or a 50-page legal contract). 

We want to ask a specific question about it. We have two options:
1. **Full Context:** Dump the entire 50-page document into the prompt.
2. **Pruned Context:** Use a naive script to extract only the first 5 pages to save money and time.

We must measure the Latency and Cost of both, and enforce a **Quality Gate**.

In [ ]:
import os
import time
from google import genai

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# Simulating a very large document
large_document = "\n".join([f"Page {i}: General legal boilerplate." for i in range(1, 49)])
large_document += "\nPage 49: The hidden clause: The company may terminate this contract at any time."
large_document += "\nPage 50: Signatures."

# The user's question
question = "Can the company terminate the contract?"
expected_answer = "yes"


## The Evaluation Harness

We define a function that runs the prompt, measures time, counts tokens, and checks the Quality Gate.

In [ ]:
def run_policy(name: str, context: str, question: str, expected_answer: str):
    print(f"\n--- Running Policy: {name} ---")
    start_time = time.time()
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=f"Answer the question based on the text. Keep the answer to a single word (yes/no).\n\nText: {context}\n\nQuestion: {question}"
    )
    
    latency = time.time() - start_time
    actual_answer = response.text.strip().lower()
    
    # Token counting for cost estimation
    tokens_used = response.usage_metadata.total_token_count if response.usage_metadata else "Unknown"
    
    print(f"Tokens Used (Cost): {tokens_used}")
    print(f"Latency: {latency:.2f} seconds")
    
    # The Quality Gate
    if expected_answer in actual_answer:
        print(f"QUALITY GATE: PASS (Answered '{actual_answer}')")
    else:
        print(f"QUALITY GATE: FAIL (Expected '{expected_answer}', got '{actual_answer}')")


## Step 1: Full Context Policy

We send the entire 50-page document.

In [ ]:
run_policy("Full Context", large_document, question, expected_answer)


## Step 2: Pruned Context Policy

To save money and reduce latency, an engineer decides to just send the first 10 pages.

In [ ]:
pruned_document = "\n".join(large_document.split("\n")[:10])
run_policy("Pruned Context", pruned_document, question, expected_answer)


## Conclusion: The Pareto Frontier

The Pruned Context policy was vastly cheaper (fewer tokens) and faster (lower latency). But it **failed the Quality Gate** because the critical information was on page 49.

Engineering is about trade-offs. You cannot blindly optimize for cost and latency without measuring the regression in quality. 

**The Modern Solution:** Instead of naive pruning, modern systems use **Context Caching**. You cache the massive `large_document` in the model's memory. This allows you to achieve the *Latency and Cost of the Pruned Policy* while maintaining the *Quality of the Full Context Policy*.